In [56]:
import pandas as pd
import duckdb
import os
import os, json
from uuid import uuid4
import pandas as pd
pd.set_option("display.max_colwidth", 200)
#pd.set_option("display.width", 500)  # adjusts total line width before wrapping
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.preprocessing import normalize
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2 import service_account # based on google-auth library
import matplotlib.pyplot as plt
import re
import unicodedata
from sklearn.preprocessing import normalize

In [57]:
try:
    file_data = json.load(open(os.path.expanduser("~/ServiceAccountsKey.json")))
    # (2) transform the content into crendentials object
    credentials = service_account.Credentials.from_service_account_info(file_data)
# (3) specify your usage of the credentials
    scoped_credentials = credentials.with_scopes(['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive'])
# (4) use the constrained credentials for authentication of gspread package
    gc = gspread.Client(auth=scoped_credentials)
    grela_gs = gc.open_by_url("https://docs.google.com/spreadsheets/d/1QroTEQ9gQf9cLO9mvolp7fELNbYYjgvGd48yAiTj03w/edit?usp=sharing")
except:
    pass

In [58]:
%%capture

# model_labse = SentenceTransformer("sentence-transformers/LaBSE")
model = SentenceTransformer("julian-schelb/multilingual-e5-large-emb-lat-intertext-v1")



In [59]:
# conn = duckdb.connect('/srv/data/grela_v0-2.duckdb', read_only=True)
conn = duckdb.connect('/srv/data/grela/grela_v0.7.duckdb', read_only=True)

In [60]:
query = """
        SELECT w.*
        FROM works w
        """
works_df = conn.execute(query).fetchdf()

In [61]:
works_df.head(5)

,grela_source,grela_id,author,title,not_before,not_after,date_random,provenience,place_publication,place_geonames,author_viaf,author_wd,author_gnd,title_viaf,subcorpus_specific_metadata,token_count,sentence_count
0,lagt,lagt_ggm0001.ggm001,Anonymous,Anametresis Pontou,1,400,157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",422,23
1,lagt,lagt_ogl0001.ogl001,Pinytus,De Epistola Pinyti ad Dionysium,101,200,174,christian,NaN,NaN,NaN,NaN,NaN,NaN,"{""lagt_tlg_epithet"":""[]"",""lagt_genre"":""[]"",""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",81,3
2,lagt,lagt_pta0001.pta001,Severian of Gabala,De fide et lege naturae,400,409,403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",4495,327
3,lagt,lagt_pta0001.pta002,Severian of Gabala,De paenitentia et compunctione,400,409,406,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",6987,539
4,lagt,lagt_pta0001.pta003,Severian of Gabala,In ascensionem domini nostri Iesu Christi et in principium Actorum,400,409,406,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",12070,1041


In [62]:
def add_cc_link(grela_id):
    if "cc_" in grela_id:
        url = "https://mlat.uzh.ch/" + grela_id.rpartition("_")[2]
        return url
works_df["cc_link"] = works_df["grela_id"].apply(add_cc_link)

In [63]:
works_df[5000:5010]

,grela_source,grela_id,author,title,not_before,not_after,date_random,provenience,place_publication,place_geonames,author_viaf,author_wd,author_gnd,title_viaf,subcorpus_specific_metadata,token_count,sentence_count,cc_link
5000,cc,cc_14346,Antonius Musa,De herba vettonica liber,350,350,350,NaN,NaN,NaN,61956528,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",1280,118,https://mlat.uzh.ch/14346
5001,cc,cc_14347,Gargilius Martialis,De hortis,260,260,260,NaN,NaN,NaN,100197892,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",2795,182,https://mlat.uzh.ch/14347
5002,cc,cc_14348,Palladius Rutilius Taurus Aemilianus,De insitione,450,450,450,NaN,NaN,NaN,54143112,Q561353,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",1321,76,https://mlat.uzh.ch/14348
5003,cc,cc_14350,Marcellus Empiricus,De medicamentis liber,395,415,404,NaN,NaN,NaN,nan,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",206666,9291,https://mlat.uzh.ch/14350
5004,cc,cc_14351,Cassius Felix,De medicina,447,447,447,NaN,NaN,NaN,61302330,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",34358,2183,https://mlat.uzh.ch/14351
5005,cc,cc_14352,Plinius Secundus Iunior Ps.,De medicina,325,325,325,NaN,NaN,NaN,nan,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",29664,675,https://mlat.uzh.ch/14352
5006,cc,cc_14353,Hyginus ps.,De metatione (uel munitionibus) castrorum,250,250,250,NaN,NaN,NaN,nan,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",4048,218,https://mlat.uzh.ch/14353
5007,cc,cc_14354,Mallius Theodorus,De metris,399,399,399,NaN,NaN,NaN,42224738,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",3669,179,https://mlat.uzh.ch/14354
5008,cc,cc_14356,Vindicianus Afer,De natura generis humani,380,380,380,NaN,NaN,NaN,17614675,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",3811,168,https://mlat.uzh.ch/14356
5009,cc,cc_14357,Martianus Capella,De nuptiis Philologiae et Mercurii,450,450,450,NaN,NaN,NaN,95152094,,NaN,NaN,"{""lagt_tlg_epithet"":null,""lagt_genre"":null,""noscemus_place"":null,""noscemus_genre"":null,""noscemus_discipline"":null,""emlap_noscemus_id"":null}",164440,9297,https://mlat.uzh.ch/14357


In [64]:
works_df.columns

Index(['grela_source', 'grela_id', 'author', 'title', 'not_before',
       'not_after', 'date_random', 'provenience', 'place_publication',
       'place_geonames', 'author_viaf', 'author_wd', 'author_gnd',
       'title_viaf', 'subcorpus_specific_metadata', 'token_count',
       'sentence_count', 'cc_link'],
      dtype='str')

In [65]:
#set_with_dataframe(grela_gs.add_worksheet("grela_works_v0-7_short", 1,1), works_df[['grela_source', 'grela_id', 'cc_link', 'author', 'title', 'not_before', 'not_after',  'token_count', 'cc_link']].sort_values("not_before", ascending=True))

In [66]:
query = """
SELECT
    s.sentence_id,
    s.grela_id,
    s.position AS sentence_position,
    s.sent_text,

    t.token_id,
    t.token_text,
    t.lemma,
    t.pos,
    t.char_start,
    t.char_end,
    t.ref,

    w.grela_source,
    w.author,
    w.title,
FROM sentences AS s
JOIN tokens AS t
    ON s.sentence_id = t.sentence_id
   AND s.grela_id = t.grela_id
LEFT JOIN works AS w
    ON s.grela_id = w.grela_id
WHERE s.grela_id LIKE 'vulgate_%' OR s.grela_id = 'cc_10265'
ORDER BY s.grela_id, s.position, t.char_start, t.token_id
"""

tokens_df = conn.execute(query).df()

In [67]:
vulgate_tokens = tokens_df[tokens_df["grela_id"].str.startswith("vulgate_")]
register_tokens = tokens_df[tokens_df["grela_id"] == "cc_10265"]

In [68]:
vulgate_tokens["chapter"] = vulgate_tokens["ref"].apply(lambda x: eval(x)["chapter"])
vulgate_tokens["verse"] = vulgate_tokens["ref"].apply(lambda x: eval(x)["verse"])

In [69]:
vulgate_df = (
    vulgate_tokens.groupby(
        ["sentence_id", "grela_id", "sentence_position", "sent_text", "author", "title"],
        dropna=False
    )
    .apply(lambda g: g[["token_id", "token_text", "lemma", "pos", "char_start", "char_end", "chapter", "verse"]].to_dict("records"))
    .reset_index(name="tokens")
)

In [70]:
vulgate_df["chapter"] = vulgate_df["tokens"].apply(lambda x: x[0]["chapter"])
vulgate_df["verse"] = vulgate_df["tokens"].apply(lambda x: x[0]["verse"])

vulgate_df.head(5)

,sentence_id,grela_id,sentence_position,sent_text,author,title,tokens,chapter,verse
0,vulgate_tlg0031.tlg001.obi-lat_0,vulgate_tlg0031.tlg001.obi-lat,0,liber generationis Iesu Christi filii David filii Abraham,Novum Testamentum,Matthew,"[{'token_id': 255096949, 'token_text': 'liber', 'lemma': 'liber', 'pos': 'ADJ', 'char_start': 0, 'char_end': 5, 'chapter': '1', 'verse': '1'}, {'token_id': 255096950, 'token_text': 'generationis',...",1,1
1,vulgate_tlg0031.tlg001.obi-lat_1,vulgate_tlg0031.tlg001.obi-lat,1,Abraham genuit Isaac Isaac autem genuit Iacob Iacob autem genuit Iudam et fratres eius,Novum Testamentum,Matthew,"[{'token_id': 255096957, 'token_text': 'Abraham', 'lemma': 'Abraham', 'pos': 'PROPN', 'char_start': 0, 'char_end': 7, 'chapter': '1', 'verse': '2'}, {'token_id': 255096958, 'token_text': 'genuit',...",1,2
2,vulgate_tlg0031.tlg001.obi-lat_10,vulgate_tlg0031.tlg001.obi-lat,10,Iosias autem genuit Iechoniam et fratres eius in transmigratione Babylonis,Novum Testamentum,Matthew,"[{'token_id': 255097078, 'token_text': 'Iosias', 'lemma': 'Iosias', 'pos': 'PROPN', 'char_start': 0, 'char_end': 6, 'chapter': '1', 'verse': '11'}, {'token_id': 255097079, 'token_text': 'autem', '...",1,11
3,vulgate_tlg0031.tlg001.obi-lat_100,vulgate_tlg0031.tlg001.obi-lat,100,beati estis cum maledixerint vobis et persecuti vos fuerint et dixerint omne malum adversum vos mentientes propter me,Novum Testamentum,Matthew,"[{'token_id': 255098504, 'token_text': 'beati', 'lemma': 'beatus', 'pos': 'ADJ', 'char_start': 0, 'char_end': 5, 'chapter': '5', 'verse': '11'}, {'token_id': 255098505, 'token_text': 'estis', 'lem...",5,11
4,vulgate_tlg0031.tlg001.obi-lat_1000,vulgate_tlg0031.tlg001.obi-lat,1000,congregatis ergo illis dixit Pilatus quem vultis dimittam vobis Barabban an Iesum qui dicitur Christus,Novum Testamentum,Matthew,"[{'token_id': 255112403, 'token_text': 'congregatis', 'lemma': 'congrego', 'pos': 'VERB', 'char_start': 0, 'char_end': 11, 'chapter': '27', 'verse': '17'}, {'token_id': 255112404, 'token_text': 'e...",27,17


In [71]:
register_tokens[1000:1010]

,sentence_id,grela_id,sentence_position,sent_text,token_id,token_text,lemma,pos,char_start,char_end,ref,grela_source,author,title
1000,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777609,illicito,illicitus,ADJ,278,286,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1001,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777610,quod,qui,PRON,287,291,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1002,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777611,contraxit,contraho,VERB,292,301,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1003,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777612,conjugio,coniugium,NOUN,302,310,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1004,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777613,aliquam,aliquis,PRON,311,318,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1005,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777614,mercedem,merces,NOUN,319,327,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1006,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777615,recipiat,recipio,VERB,328,336,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1007,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777616,qua,qui,PRON,337,340,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1008,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777617,laetari,laeto,VERB,341,348,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum
1009,cc_10265_57,cc_10265,57,Tua itaque fraternitas consanguinitatis lineam a majoribus natu ejusdem loci diligenter inquirat quam si inter defunctum virum eamdemque vivam mulierem invenerit ad aliarum exemplum ita studeat ca...,176777618,possit,possum,VERB,349,355,"{""div_pid"": ""10265:5"", ""parent_pid"": ""10265:5;4"", ""sent_n"": ""3""}",cc,Gregorius VII,Registrum


In [72]:
register_tokens["register_ref"] = register_tokens["ref"].apply(lambda x: eval(x)["parent_pid"].rpartition(":")[2])

In [73]:
register_df = (
    register_tokens.groupby(
        ["sentence_id", "grela_id", "sentence_position", "sent_text", "author", "title"],
        dropna=False
    )
    .apply(lambda g: g[["token_id", "token_text", "lemma", "pos", "char_start", "char_end", "register_ref"]].to_dict("records"))
    .reset_index(name="tokens")
)

In [74]:
register_df["register_ref"] = register_df["tokens"].apply(lambda x: x[0]["register_ref"])


In [75]:
register_df["sentence_position"] = register_df["sentence_position"].astype(int)

In [76]:
register_df.sort_values("sentence_position", inplace=True)
register_df.head(5)

,sentence_id,grela_id,sentence_position,sent_text,author,title,tokens,register_ref
0,cc_10265_0,cc_10265,0,Registri Liber Primus,Gregorius VII,Registrum,"[{'token_id': 176776609, 'token_text': 'REGISTRI', 'lemma': 'registri', 'pos': 'PROPN', 'char_start': 0, 'char_end': 8, 'register_ref': '1;1'}, {'token_id': 176776610, 'token_text': 'LIBER', 'lemm...",1;1
1,cc_10265_1,cc_10265,1,Epistola Prima,Gregorius VII,Registrum,"[{'token_id': 176776612, 'token_text': 'EPISTOLA', 'lemma': 'epistola', 'pos': 'NOUN', 'char_start': 0, 'char_end': 8, 'register_ref': '1;2'}, {'token_id': 176776613, 'token_text': 'PRIMA', 'lemma...",1;2
1112,cc_10265_2,cc_10265,2,AD Desiderium Abbatem,Gregorius VII,Registrum,"[{'token_id': 176776614, 'token_text': 'AD', 'lemma': 'ad', 'pos': 'ADP', 'char_start': 0, 'char_end': 2, 'register_ref': '1;2'}, {'token_id': 176776615, 'token_text': 'DESIDERIUM', 'lemma': 'desi...",1;2
2223,cc_10265_3,cc_10265,3,Nuntiat se invitum in demortui Alexandri pontificis locum suffectum,Gregorius VII,Registrum,"[{'token_id': 176776617, 'token_text': 'Nuntiat', 'lemma': 'nuntio', 'pos': 'VERB', 'char_start': 0, 'char_end': 7, 'register_ref': '1;3'}, {'token_id': 176776618, 'token_text': 'se', 'lemma': 'se...",1;3
3334,cc_10265_4,cc_10265,4,Rogat ut Deum pro se deprecetur et ad se quantocius veniat,Gregorius VII,Registrum,"[{'token_id': 176776626, 'token_text': 'Rogat', 'lemma': 'rogo', 'pos': 'VERB', 'char_start': 0, 'char_end': 5, 'register_ref': '1;3'}, {'token_id': 176776627, 'token_text': 'ut', 'lemma': 'ut', '...",1;3


In [77]:
def preprocess_sentence(sentence):
    # Normalize Unicode (optional, for accented chars)
    sentence = unicodedata.normalize("NFD", sentence)
    # Remove all non-alphabetic characters (keep spaces)
    sentence = re.sub(r"[^a-zA-Z\s]", "", sentence)
    # Lowercase and Latin orthographic normalization
    sentence = sentence.lower().replace("v", "u").replace("j", "i")
    return sentence

In [78]:
vulgate_df["sent_text_clean"] = vulgate_df["sent_text"].apply(preprocess_sentence)
register_df["sent_text_clean"] = register_df["sent_text"].apply(preprocess_sentence)

In [79]:
vulgate_df.to_parquet("../data/large_files/vulgate_df.parquet")

In [80]:
all_embeddings = []
all_ids = []   # change to 'sentence_id' column if you have it, else use df.index

batch_size = 256
device = "cuda"

for i in range(0, len(vulgate_df), batch_size):
    batch = vulgate_df.iloc[i:i + batch_size].copy()
    print(f"Encoding batch {i}–{i + len(batch)}")

    emb = model.encode(batch["sent_text_clean"].tolist(), convert_to_numpy=True, device=device)

    # ensure float32
    emb = emb.astype('float32')

    all_embeddings.append(emb)
    # store row identifiers so you can map back later
    if "sentence_id" in batch.columns:
        all_ids.extend(batch["sentence_id"].tolist())
    else:
        all_ids.extend(batch.index.tolist())

# stack into a single numpy array
emb_matrix = np.vstack(all_embeddings)  # shape (N, dim)
print("Final embeddings shape:", emb_matrix.shape)


Encoding batch 0–256
Encoding batch 256–512
Encoding batch 512–768
Encoding batch 768–1024
Encoding batch 1024–1280
Encoding batch 1280–1536
Encoding batch 1536–1792
Encoding batch 1792–2048
Encoding batch 2048–2304
Encoding batch 2304–2560
Encoding batch 2560–2816
Encoding batch 2816–3072
Encoding batch 3072–3328
Encoding batch 3328–3584
Encoding batch 3584–3840
Encoding batch 3840–4096
Encoding batch 4096–4352
Encoding batch 4352–4608
Encoding batch 4608–4864
Encoding batch 4864–5120
Encoding batch 5120–5376
Encoding batch 5376–5632
Encoding batch 5632–5888
Encoding batch 5888–6144
Encoding batch 6144–6400
Encoding batch 6400–6656
Encoding batch 6656–6912
Encoding batch 6912–7168
Encoding batch 7168–7424
Encoding batch 7424–7680
Encoding batch 7680–7936
Encoding batch 7936–8192
Encoding batch 8192–8448
Encoding batch 8448–8704
Encoding batch 8704–8960
Encoding batch 8960–9216
Encoding batch 9216–9472
Encoding batch 9472–9728
Encoding batch 9728–9984
Encoding batch 9984–10240
Encoding

In [81]:
# optionally L2-normalize now (useful for cosine with IndexFlatIP)
#from sklearn.preprocessing import normalize
emb_matrix = normalize(emb_matrix, norm='l2').astype('float32')

# save compressed file for reuse
np.savez_compressed("../data/large_files/vulgate_embeddings.npz", embeddings=emb_matrix, ids=np.array(all_ids))

In [82]:
emb_matrix

array([[ 0.01940878, -0.03173655, -0.02698406, ...,  0.01017355,
        -0.02606423, -0.0039858 ],
       [ 0.02200356, -0.01574731,  0.00038979, ..., -0.00202234,
        -0.02366604, -0.01451501],
       [ 0.04171255, -0.02282952, -0.02550124, ..., -0.01413793,
        -0.04117054, -0.00307051],
       ...,
       [ 0.02783588,  0.01451133, -0.04151956, ...,  0.01992665,
        -0.01378865, -0.00930116],
       [ 0.05875102,  0.01418975, -0.01250934, ...,  0.01197144,
        -0.00400059,  0.00403252],
       [ 0.02389816, -0.00889452, -0.02111818, ...,  0.01155517,
        -0.00795731, -0.00797166]], shape=(35254, 1024), dtype=float32)

In [83]:
emb_matrix = np.load("../data/large_files/vulgate_embeddings.npz", allow_pickle=True)["embeddings"]

In [84]:
d = emb_matrix.shape[1]  # 768
index = faiss.IndexFlatIP(d)        # Inner product = cosine if normalized
index.add(emb_matrix)

In [85]:
query = "The Word became flesh and made his dwelling among us. We have seen his glory, the glory of the one and only Son, who came from the Father, full of grace and truth."
embedding = model.encode([query], convert_to_numpy=True)
# Normalize for cosine similarity
embedding = normalize(embedding, norm='l2')

In [86]:
k = 5  # number of nearest neighbors
scores, indices = index.search(embedding, k)

In [87]:
import numpy as np
# scores, indices from: scores, indices = index.search(embedding, k)
top_idxs = indices[0].astype(int)         # shape (k,)
top_scores = scores[0].astype(float)      # shape (k,)

# Get the corresponding rows directly
matches_df = vulgate_df.iloc[top_idxs].copy().reset_index(drop=True)
matches_df["score"] = top_scores


In [88]:
matches_df

,sentence_id,grela_id,sentence_position,sent_text,author,title,tokens,chapter,verse,sent_text_clean,score
0,vulgate_tlg0031.tlg004.obi-lat_13,vulgate_tlg0031.tlg004.obi-lat,13,et Verbum caro factum est et habitavit in nobis et vidimus gloriam eius gloriam quasi unigeniti a Patre plenum gratiae et veritatis,Johnannine literature,John,"[{'token_id': 255344706, 'token_text': 'et', 'lemma': 'et', 'pos': 'CCONJ', 'char_start': 0, 'char_end': 2, 'chapter': '1', 'verse': '14'}, {'token_id': 255344707, 'token_text': 'Verbum', 'lemma':...",1,14,et uerbum caro factum est et habitauit in nobis et uidimus gloriam eius gloriam quasi unigeniti a patre plenum gratiae et ueritatis,0.794845
1,vulgate_tlg0031.tlg005.obi-lat_247,vulgate_tlg0031.tlg005.obi-lat,247,cum autem esset plenus Spiritu Sancto intendens in caelum vidit gloriam Dei et Iesum stantem a dextris Dei et ait ecce video caelos apertos et Filium hominis a dextris stantem Dei,Luke (the evangelist),Acts,"[{'token_id': 255005407, 'token_text': 'cum', 'lemma': 'cum', 'pos': 'ADP', 'char_start': 0, 'char_end': 3, 'chapter': '7', 'verse': '55'}, {'token_id': 255005408, 'token_text': 'autem', 'lemma': ...",7,55,cum autem esset plenus spiritu sancto intendens in caelum uidit gloriam dei et iesum stantem a dextris dei et ait ecce uideo caelos apertos et filium hominis a dextris stantem dei,0.752596
2,vulgate_tlg0031.tlg010.obi-lat_78,vulgate_tlg0031.tlg010.obi-lat,78,donec occurramus omnes in unitatem fidei et agnitionis Filii Dei in virum perfectum in mensuram aetatis plenitudinis Christi,Pauline literature,Ephesians,"[{'token_id': 255267117, 'token_text': 'donec', 'lemma': 'donec', 'pos': 'SCONJ', 'char_start': 0, 'char_end': 5, 'chapter': '4', 'verse': '13'}, {'token_id': 255267118, 'token_text': 'occurramus'...",4,13,donec occurramus omnes in unitatem fidei et agnitionis filii dei in uirum perfectum in mensuram aetatis plenitudinis christi,0.752360
3,vulgate_tlg0031.tlg023.obi-lat_71,vulgate_tlg0031.tlg023.obi-lat,71,in hoc apparuit caritas Dei in nobis quoniam Filium suum unigenitum misit Deus in mundum ut vivamus per eum,Johnannine literature,1 John,"[{'token_id': 255033170, 'token_text': 'in', 'lemma': 'in', 'pos': 'ADP', 'char_start': 0, 'char_end': 2, 'chapter': '4', 'verse': '9'}, {'token_id': 255033171, 'token_text': 'hoc', 'lemma': 'hic'...",4,9,in hoc apparuit caritas dei in nobis quoniam filium suum unigenitum misit deus in mundum ut uiuamus per eum,0.749398
4,vulgate_tlg0031.tlg019.obi-lat_37,vulgate_tlg0031.tlg019.obi-lat,37,Christus vero tamquam filius in domo sua quae domus sumus nos si fiduciam et gloriam spei usque ad finem firmam retineamus,Novum Testamentum,Hebrews,"[{'token_id': 254996658, 'token_text': 'Christus', 'lemma': 'Christus', 'pos': 'PROPN', 'char_start': 0, 'char_end': 8, 'chapter': '3', 'verse': '6'}, {'token_id': 254996659, 'token_text': 'vero',...",3,6,christus uero tamquam filius in domo sua quae domus sumus nos si fiduciam et gloriam spei usque ad finem firmam retineamus,0.739645


In [89]:
len(register_df)

5399

In [90]:
all_embeddings = []
all_ids = []   # change to 'sentence_id' column if you have it, else use df.index

batch_size = 256
device = "cuda"

for i in range(0, len(register_df), batch_size):
    batch = register_df.iloc[i:i + batch_size].copy()
    print(f"Encoding batch {i}–{i + len(batch)}")

    emb = model.encode(batch["sent_text_clean"].tolist(), convert_to_numpy=True, device=device)

    # ensure float32
    emb = emb.astype('float32')

    all_embeddings.append(emb)
    # store row identifiers so you can map back later
    if "sentence_id" in batch.columns:
        all_ids.extend(batch["sentence_id"].tolist())
    else:
        all_ids.extend(batch.index.tolist())

# stack into a single numpy array
emb_matrix = np.vstack(all_embeddings)  # shape (N, dim)
print("Final embeddings shape:", emb_matrix.shape)

Encoding batch 0–256
Encoding batch 256–512
Encoding batch 512–768
Encoding batch 768–1024
Encoding batch 1024–1280
Encoding batch 1280–1536
Encoding batch 1536–1792
Encoding batch 1792–2048
Encoding batch 2048–2304
Encoding batch 2304–2560
Encoding batch 2560–2816
Encoding batch 2816–3072
Encoding batch 3072–3328
Encoding batch 3328–3584
Encoding batch 3584–3840
Encoding batch 3840–4096
Encoding batch 4096–4352
Encoding batch 4352–4608
Encoding batch 4608–4864
Encoding batch 4864–5120
Encoding batch 5120–5376
Encoding batch 5376–5399
Final embeddings shape: (5399, 1024)


In [91]:
emb_matrix = normalize(emb_matrix, norm='l2').astype('float32')

In [92]:
register_df["embedding"] = emb_matrix.tolist()

In [93]:
register_df[13:25]

,sentence_id,grela_id,sentence_position,sent_text,author,title,tokens,register_ref,sent_text_clean,embedding
335,cc_10265_13,cc_10265,13,Te itaque per omnipotentem Dominum rogo ut suffraganeos fratres et filios quos in Christo nutris ad exorandum Deum pro me provoces et ex vera charitate invites quatenus oratio quae me liberare deb...,Gregorius VII,Registrum,"[{'token_id': 176776798, 'token_text': 'Te', 'lemma': 'tu', 'pos': 'PRON', 'char_start': 0, 'char_end': 2, 'register_ref': '1;5'}, {'token_id': 176776799, 'token_text': 'itaque', 'lemma': 'itaque'...",1;5,te itaque per omnipotentem dominum rogo ut suffraganeos fratres et filios quos in christo nutris ad exorandum deum pro me prouoces et ex uera charitate inuites quatenus oratio quae me liberare deb...,"[0.008061675354838371, 0.0005415214109234512, -0.04581218585371971, 0.00038499085349030793, 0.0056756935082376, -0.035338521003723145, 0.007762026973068714, 0.06096648424863815, 0.0102162249386310..."
446,cc_10265_14,cc_10265,14,Tu autem ipse quantocius ad nos venire non praetermittas qui quantum Romana Ecclesia te indigeat et in prudentia tua fiduciam habeat non ignoras,Gregorius VII,Registrum,"[{'token_id': 176776838, 'token_text': 'Tu', 'lemma': 'tu', 'pos': 'PRON', 'char_start': 0, 'char_end': 2, 'register_ref': '1;5'}, {'token_id': 176776839, 'token_text': 'autem', 'lemma': 'autem', ...",1;5,tu autem ipse quantocius ad nos uenire non praetermittas qui quantum romana ecclesia te indigeat et in prudentia tua fiduciam habeat non ignoras,"[0.008015401661396027, -0.017786744982004166, -0.03930177539587021, -0.004931995645165443, -0.006586124654859304, -0.013376948423683643, 0.021429797634482384, -0.009945015422999859, -0.00459344405..."
557,cc_10265_15,cc_10265,15,Dominam Agnetem imperatricem et Rainaldum venerabilem Cumanum episcopum ex nostra parte saluta et quantum erga nos dilectionis habuerint nunc ut ostendant nostra vice fideliter obsecra,Gregorius VII,Registrum,"[{'token_id': 176776861, 'token_text': 'Dominam', 'lemma': 'domina', 'pos': 'NOUN', 'char_start': 0, 'char_end': 7, 'register_ref': '1;5'}, {'token_id': 176776862, 'token_text': 'Agnetem', 'lemma'...",1;5,dominam agnetem imperatricem et rainaldum uenerabilem cumanum episcopum ex nostra parte saluta et quantum erga nos dilectionis habuerint nunc ut ostendant nostra uice fideliter obsecra,"[-0.005316360853612423, 0.0051057119853794575, -0.02686578594148159, -0.011614331044256687, 0.01529298909008503, -0.025296613574028015, -0.01705058105289936, 0.02250220999121666, 0.024227758869528..."
668,cc_10265_16,cc_10265,16,Data Datum et sic semper Romae XI IX,Gregorius VII,Registrum,"[{'token_id': 176776886, 'token_text': 'Data', 'lemma': 'do', 'pos': 'VERB', 'char_start': 0, 'char_end': 4, 'register_ref': '1;6'}, {'token_id': 176776887, 'token_text': 'Datum', 'lemma': '-', 'p...",1;6,data datum et sic semper romae xi ix,"[0.005302080884575844, -0.004149687942117453, -0.028951477259397507, -0.00996397715061903, 0.03109694831073284, -0.04280992969870567, -0.001994697144255042, 0.04297688603401184, 0.0735568106174469..."
779,cc_10265_17,cc_10265,17,Kalendas Maii indictione XI,Gregorius VII,Registrum,"[{'token_id': 176776894, 'token_text': 'Kalendas', 'lemma': 'Kalenda', 'pos': 'NOUN', 'char_start': 0, 'char_end': 8, 'register_ref': '1;6'}, {'token_id': 176776895, 'token_text': 'Maii', 'lemma':...",1;6,kalendas maii indictione xi,"[0.04913956671953201, 0.004739556927233934, -0.01652904972434044, -0.009477960877120495, 0.031295422464609146, -0.011601502075791359, -0.025919117033481598, 0.05671605467796326, 0.0502603501081466..."
890,cc_10265_18,cc_10265,18,Epistola II,Gregorius VII,Registrum,"[{'token_id': 176776898, 'token_text': 'EPISTOLA', 'lemma': 'epistola', 'pos': 'NOUN', 'char_start': 0, 'char_end': 8, 'register_ref': '2;1'}, {'token_id': 176776899, 'token_text': 'II', 'lemma': ...",2;1,epistola ii,"[0.009238982573151588, 0.01454919669777155, 0.0027062196750193834, -0.030291888862848282, -0.008128981105983257, -0.05

In [94]:
register_df.to_parquet("../data/large_files/register_df_with_embeddings.parquet")